# Game Theory: Minimax Algorithm and Alpha-Beta Pruning

## 1. Introduction

Game theory studies decision-making situations in which the outcome for one participant depends on the choices made by other participants. In competitive or adversarial settings, each participant may choose a strategy to improve their own outcome.

In Artificial Intelligence, game trees represent the possible sequences of decisions. The **Minimax algorithm** selects a move by assuming that one player tries to maximise the utility while the opponent tries to minimise it. **Alpha-Beta pruning** improves Minimax search by avoiding branches that cannot affect the final decision.

This notebook uses the original example of strategic price competition between two companies. Company A is treated as the MAX player and Company B as the MIN player. The leaf values represent Company A's profit.

## 2. Aim

To understand game-tree representation and implement the Minimax algorithm and Alpha-Beta pruning using Python.

## 3. Learning Objectives

- Explain the role of game theory in strategic decision-making.
- Represent a sequential decision problem as a game tree.
- Identify MAX and MIN players and interpret utility values.
- Implement the Minimax algorithm recursively.
- Implement Alpha-Beta pruning using alpha and beta bounds.
- Compare the results and search effort of both algorithms.

## 4. Problem Statement

Two companies, A and B, compete by choosing either a Low Price or High Price strategy. Company A chooses first, followed by Company B. The resulting leaf value is Company A's profit.

Company A aims to maximise its profit (MAX), while Company B aims to minimise Company A's profit (MIN). Determine the optimal payoff for Company A using Minimax and Alpha-Beta pruning, and compare the algorithms.


## 5. Theory: Game Theory and Adversarial Search

### 5.1 Game theory

Game theory provides mathematical models for situations involving strategic interaction between decision-makers. In a two-player, zero-sum setting, one player's gain corresponds to the other player's loss under the chosen utility measure.

### 5.2 Game tree

A game tree is a tree-shaped representation of possible decisions. Each internal node represents a decision point, each edge represents a possible action, and each leaf contains a terminal utility or payoff.

### 5.3 MAX and MIN players

- **MAX player:** selects the action with the highest backed-up value.
- **MIN player:** selects the action with the lowest backed-up value.

The players are assumed to alternate turns. In this example, Company A is MAX at the root and Company B is MIN at the next level.

### 5.4 Minimax algorithm

Minimax evaluates a game tree from the leaves toward the root. At a MAX node it returns the maximum child value; at a MIN node it returns the minimum child value. The root value is the outcome that MAX can guarantee when both players play optimally.

For a node \(n\):

\[
V(n)=
\begin{cases}
\max\limits_{c \in Children(n)} V(c), & \text{if } n \text{ is MAX}\\
\min\limits_{c \in Children(n)} V(c), & \text{if } n \text{ is MIN}
\end{cases}
\]

### 5.5 Alpha-Beta pruning

Alpha-Beta pruning is an optimisation of Minimax. It returns the same minimax value but may examine fewer nodes.

- **Alpha (α):** the best value MAX can guarantee along the current path.
- **Beta (β):** the best value MIN can guarantee along the current path.
- When \(\alpha \geq \beta\), the remaining children of that node cannot influence the final decision and can be pruned.

The effectiveness of pruning depends on the order in which branches are explored. In the example below, the children are explored in the order in which they appear in the dictionary.

### 5.6 Complexity

In the worst case, Alpha-Beta pruning may examine as many nodes as Minimax. With favourable move ordering, it can substantially reduce the search effort. The theoretical best-case improvement is often described as reducing the effective search from approximately \(O(b^d)\) to \(O(b^{d/2})\), where \(b\) is the branching factor and \(d\) is the search depth. The exact benefit depends on the tree and ordering.


## 6. Representing the Economic Game Tree

The original payoff structure is preserved:

| Company A strategy | Company B: Low Price | Company B: High Price |
|---|---:|---:|
| Low Price | 8 | 12 |
| High Price | 4 | 10 |

Each payoff is Company A's profit. Company A chooses the row first; Company B then chooses the response. The nested dictionary below represents the same game tree.


In [ ]:
# Company A is MAX and moves first.
# Company B is MIN and responds.
# Leaf values represent Company A's profit.

game_tree = {
    "Low Price": {
        "Low Price": 8,
        "High Price": 12
    },
    "High Price": {
        "Low Price": 4,
        "High Price": 10
    }
}

print("Economic Game Tree")
print("=" * 45)

for company_a_strategy, responses in game_tree.items():
    print(f"Company A chooses: {company_a_strategy}")
    for company_b_strategy, payoff in responses.items():
        print(f"  Company B chooses: {company_b_strategy}")
        print(f"  Profit of Company A: {payoff}")

## 7. Understanding the Payoffs

Company B is assumed to respond in a way that minimises Company A's profit.

- If Company A chooses **Low Price**, Company B can hold Company A's payoff to the minimum of 8 and 12, which is 8.
- If Company A chooses **High Price**, Company B can hold Company A's payoff to the minimum of 4 and 10, which is 4.
- Company A then chooses the strategy with the larger guaranteed payoff.

This reasoning can be performed manually and will be verified by the algorithms.


In [ ]:
# Calculate the value of each Company A strategy assuming
# Company B chooses the response least favourable to Company A.

strategy_values = {
    strategy: min(responses.values())
    for strategy, responses in game_tree.items()
}

print("Company A strategy values under optimal Company B responses:")
for strategy, value in strategy_values.items():
    print(f"{strategy}: {value}")

manual_best_strategy = max(strategy_values, key=strategy_values.get)
manual_best_value = strategy_values[manual_best_strategy]

print("\nBest strategy for Company A:", manual_best_strategy)
print("Guaranteed payoff:", manual_best_value)

## 8. Implementing the Minimax Algorithm

The recursive function checks whether the current node is a terminal payoff. If it is, the payoff is returned. Otherwise, the function evaluates every child and returns either the maximum or minimum value depending on whose turn it is.

A counter is included to record how many terminal payoff nodes are evaluated.


In [ ]:
def minimax(node, maximizing, stats=None):
    """Return the Minimax value of a nested-dictionary game tree."""
    if stats is None:
        stats = {"leaf_evaluations": 0}

    if isinstance(node, (int, float)):
        stats["leaf_evaluations"] += 1
        return node

    if not isinstance(node, dict) or not node:
        raise ValueError("Each non-terminal node must be a non-empty dictionary.")

    child_values = [
        minimax(child, not maximizing, stats)
        for child in node.values()
    ]

    return max(child_values) if maximizing else min(child_values)


minimax_stats = {"leaf_evaluations": 0}
minimax_result = minimax(game_tree, maximizing=True, stats=minimax_stats)

print("Minimax optimal payoff for Company A:", minimax_result)
print("Terminal payoff nodes evaluated:", minimax_stats["leaf_evaluations"])

## 9. Step-by-Step Minimax Evaluation

The leaf payoffs are evaluated first. Company B (MIN) chooses the smaller payoff in each row. Company A (MAX) then chooses the larger of those two results.


In [ ]:
print("MIN level: Company B's best responses for Company A")
for strategy, value in strategy_values.items():
    print(f"{strategy}: min payoff = {value}")

print("\nMAX level: Company A chooses the larger guaranteed payoff")
print(f"max({', '.join(str(v) for v in strategy_values.values())}) = {minimax_result}")

## 10. Implementing Alpha-Beta Pruning

The Alpha-Beta function follows the same MAX/MIN logic as Minimax while maintaining alpha and beta bounds. When the bounds cross, the remaining children at that node are skipped.

The statistics record:
- Terminal leaves actually evaluated.
- Terminal leaves skipped because of pruning.

The implementation uses the same child ordering as the original game tree.


In [ ]:
def alpha_beta(node, alpha, beta, maximizing, stats=None):
    """Return the Alpha-Beta value while counting evaluated and pruned leaves."""
    if stats is None:
        stats = {"leaf_evaluations": 0, "pruned_leaves": 0}

    if isinstance(node, (int, float)):
        stats["leaf_evaluations"] += 1
        return node

    if not isinstance(node, dict) or not node:
        raise ValueError("Each non-terminal node must be a non-empty dictionary.")

    children = list(node.values())

    if maximizing:
        best_value = float("-inf")

        for index, child in enumerate(children):
            value = alpha_beta(child, alpha, beta, False, stats)
            best_value = max(best_value, value)
            alpha = max(alpha, best_value)

            if alpha >= beta:
                # Count terminal payoffs in children that will not be visited.
                stats["pruned_leaves"] += sum(count_leaves(c) for c in children[index + 1:])
                break

        return best_value

    best_value = float("inf")

    for index, child in enumerate(children):
        value = alpha_beta(child, alpha, beta, True, stats)
        best_value = min(best_value, value)
        beta = min(beta, best_value)

        if alpha >= beta:
            stats["pruned_leaves"] += sum(count_leaves(c) for c in children[index + 1:])
            break

    return best_value


def count_leaves(node):
    """Count terminal payoff values below a node without evaluating them."""
    if isinstance(node, (int, float)):
        return 1
    return sum(count_leaves(child) for child in node.values())


alpha_beta_stats = {"leaf_evaluations": 0, "pruned_leaves": 0}
alpha_beta_result = alpha_beta(
    game_tree,
    alpha=float("-inf"),
    beta=float("inf"),
    maximizing=True,
    stats=alpha_beta_stats
)

print("Alpha-Beta optimal payoff for Company A:", alpha_beta_result)
print("Terminal payoff nodes evaluated:", alpha_beta_stats["leaf_evaluations"])
print("Terminal payoff nodes pruned:", alpha_beta_stats["pruned_leaves"])

## 11. Understanding the Pruning in This Game

With the given branch order:

1. Company A's **Low Price** branch is explored first. Company B's minimum payoff in that branch is 8. Therefore, MAX has an alpha value of 8.
2. The search then enters Company A's **High Price** branch. Company B first considers Low Price, producing a payoff of 4. This sets beta to 4.
3. Since beta (4) is now less than or equal to alpha (8), the remaining response under this MIN node cannot improve the result for MAX. The final payoff of 10 in that branch is pruned.

The pruned branch is not needed to determine the root's Minimax value.


In [ ]:
print("Pruning summary")
print("-" * 40)
print("Alpha after first Company A branch: 8")
print("Beta after first response in second branch: 4")
print("Pruning condition: alpha >= beta")
print("Condition satisfied:", 8 >= 4)
print("Skipped payoff in this tree: 10")

## 12. Comparing Minimax and Alpha-Beta Pruning

Both algorithms must return the same optimal payoff. The difference is the amount of terminal payoff information they need to evaluate.

For this four-leaf tree, Minimax evaluates every leaf. With the current branch order, Alpha-Beta evaluates three leaves and prunes one.


In [ ]:
print("ALGORITHM COMPARISON")
print("=" * 50)
print(f"Minimax result:              {minimax_result}")
print(f"Alpha-Beta result:           {alpha_beta_result}")
print(f"Minimax leaves evaluated:    {minimax_stats['leaf_evaluations']}")
print(f"Alpha-Beta leaves evaluated: {alpha_beta_stats['leaf_evaluations']}")
print(f"Leaves pruned:               {alpha_beta_stats['pruned_leaves']}")

assert minimax_result == alpha_beta_result
print("\nBoth algorithms return the same optimal payoff.")

## 13. Automated Test Cases

The following tests verify the known result, the equivalence of both algorithms, the number of evaluated leaves for this branch order, and the correctness of the pruning count.


In [ ]:
# Test 1: Manually derived strategy values.
assert strategy_values == {"Low Price": 8, "High Price": 4}

# Test 2: The best strategy and payoff for Company A.
assert manual_best_strategy == "Low Price"
assert manual_best_value == 8

# Test 3: Minimax and Alpha-Beta must agree.
assert minimax_result == alpha_beta_result == 8

# Test 4: Minimax evaluates every terminal payoff.
assert minimax_stats["leaf_evaluations"] == 4

# Test 5: Alpha-Beta evaluates three leaves and prunes one
# for the current ordering of the branches.
assert alpha_beta_stats["leaf_evaluations"] == 3
assert alpha_beta_stats["pruned_leaves"] == 1

# Test 6: A reversed branch order may change pruning efficiency,
# but it must not change the minimax value.
reversed_tree = {
    "High Price": game_tree["High Price"],
    "Low Price": game_tree["Low Price"]
}
assert minimax(reversed_tree, True) == 8
assert alpha_beta(reversed_tree, float("-inf"), float("inf"), True) == 8

print("All six automated tests passed successfully.")

## 14. Observations

1. Game trees represent sequential decisions and their possible outcomes.
2. Company A acts as MAX because it aims to maximise its profit.
3. Company B acts as MIN because it aims to minimise Company A's profit.
4. The Minimax result is 8, so Company A's best strategy under the stated assumptions is Low Price.
5. Alpha-Beta pruning returns the same result as Minimax.
6. With the original branch order, Alpha-Beta evaluates three of the four terminal payoffs and prunes one.
7. Reordering branches can change the number of pruned nodes without changing the correct Minimax value.

## 15. Limitations

- The example is a small, deterministic, two-player game with complete information.
- Payoffs are treated as fixed values and represent only Company A's profit.
- The model assumes both companies choose optimally.
- The example does not include simultaneous decisions, uncertainty, repeated interactions, or changing market conditions.
- Alpha-Beta pruning is demonstrated on a small tree; larger trees are needed to show its full search-efficiency benefits.

## 16. Conclusion

This notebook demonstrates game-tree representation, Minimax decision-making, and Alpha-Beta pruning through a price-competition example. Company A is the MAX player and Company B is the MIN player. The Minimax algorithm determines that Company A can guarantee a payoff of 8 by choosing Low Price.

Alpha-Beta pruning produces the same optimal payoff while avoiding evaluation of one terminal branch in the given branch order. The experiment demonstrates that pruning can reduce search effort without changing the decision selected by Minimax.

## 17. References

1. Russell, S. J., and Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson.
2. Goodfellow, I., Bengio, Y., and Courville, A. (2016). *Deep Learning*. MIT Press. (Background on optimisation and search concepts.)
3. MIT OpenCourseWare, *Game Playing and Minimax Search* lecture materials: https://ocw.mit.edu/
